# Clause Classification using Machine Learning

## Objective
Automatically classify legal clauses into predefined categories
(e.g., Termination, Liability, Confidentiality).

In this notebook:
- We train a baseline ML model
- Use TF-IDF for text representation
- Use Logistic Regression for classification


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


In [2]:
# Load clause-level dataset
df = pd.read_csv("../data/processed/cuad_segmented_clauses.csv")

df.head()


,file_name,clause,pages,class_id,label,start_at,end_at,clean_text,sentences,word_count,clause_text
0,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,In the event that Licensor grants to another V...,2,8,Most Favored Nation,2558,2929,in the event that licensor grants to another v...,['in the event that licensor grants to another...,60,in the event that licensor grants to another v...
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"If Licensor enters, or has entered, into an ag...",8,8,Most Favored Nation,18515,19562,if licensor enters or has entered into an agre...,['if licensor enters or has entered into an ag...,169,if licensor enters or has entered into an agre...
2,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"Licensor shall provide to Rogers, no later tha...",8,8,Most Favored Nation,19563,20059,licensor shall provide to rogers no later than...,['licensor shall provide to rogers no later th...,79,licensor shall provide to rogers no later than...
3,IntegrityMediaInc_20010329_10-K405_EX-10.17_23...,"If for any reason, Integrity and TL are subjec...",3,8,Most Favored Nation,8173,8345,if for any reason integrity and tl are subject...,['if for any reason integrity and tl are subje...,30,if for any reason integrity and tl are subject...
4,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,"The Company will, and Online BVI will cause th...",10,8,Most Favored Nation,30765,31577,the company will and online bvi will cause the...,['the company will and online bvi will cause t...,123,the company will and online bvi will cause the...


In [3]:
print("Total clauses:", len(df))
print("\nTop clause types:")
print(df["label"].value_counts().head(10))


Total clauses: 9698

Top clause types:
label
License Grant                721
Cap On Liability             636
Anti-Assignment              615
Audit Rights                 615
Insurance                    532
Governing Law                455
Expiration Date              451
Post-Termination Services    411
Minimum Commitment           401
Revenue/Profit Sharing       397
Name: count, dtype: int64


In [4]:
top_labels = df["label"].value_counts().head(6).index.tolist()

df = df[df["label"].isin(top_labels)]

print("Remaining labels:", df["label"].unique())
print("Dataset size:", len(df))


Remaining labels: ['Anti-Assignment' 'Audit Rights' 'Insurance' 'License Grant'
 'Cap On Liability' 'Governing Law']
Dataset size: 3574


In [5]:
X = df["clause_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf.shape


(2859, 5000)

In [7]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)


LogisticRegression(max_iter=1000, n_jobs=-1)

In [8]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.9664335664335665

Classification Report:

                  precision    recall  f1-score   support

 Anti-Assignment       0.97      0.93      0.95       123
    Audit Rights       0.98      0.97      0.97       123
Cap On Liability       0.96      0.98      0.97       127
   Governing Law       1.00      0.99      0.99        91
       Insurance       0.99      0.96      0.98       107
   License Grant       0.92      0.97      0.95       144

        accuracy                           0.97       715
       macro avg       0.97      0.97      0.97       715
    weighted avg       0.97      0.97      0.97       715



In [10]:
sample_clauses = [
    "Either party may terminate this agreement with thirty days written notice.",
    "The company shall not be liable for any indirect or consequential damages.",
    "All confidential information must be kept secret."
]

sample_tfidf = tfidf.transform(sample_clauses)
predictions = model.predict(sample_tfidf)

for clause, label in zip(sample_clauses, predictions):
    print(f"\nClause: {clause}")
    print(f"Predicted Type: {label}")



Clause: Either party may terminate this agreement with thirty days written notice.
Predicted Type: Anti-Assignment

Clause: The company shall not be liable for any indirect or consequential damages.
Predicted Type: Cap On Liability

Clause: All confidential information must be kept secret.
Predicted Type: Audit Rights


In [11]:
import os
import pickle

os.makedirs("../models/clause_classifier", exist_ok=True)

with open("../models/clause_classifier/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("../models/clause_classifier/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model and vectorizer saved successfully.")


Model and vectorizer saved successfully.


## Summary

In this notebook, we:
- Trained a baseline clause classification model
- Used TF-IDF for text representation
- Applied Logistic Regression for multi-class classification
- Evaluated performance using standard metrics
- Saved trained models for deployment

Next notebook:
➡ 04_risk_analysis.ipynb
